# LAB 2 - Creating SQL Schema

### Step 1: Importing Libraries

In [19]:
import duckdb
import os
import kagglehub
import pandas as pd
from google.colab import files

### Step 1a: Loading the dataset

In [7]:
# Download latest version of the dataset
path = kagglehub.dataset_download("prasad22/healthcare-dataset")
print("Path to dataset files:", path)

# List files in the downloaded dataset folder
print(os.listdir(path))
df = pd.read_csv(os.path.join(path, "healthcare_dataset.csv"))
df.head()


Using Colab cache for faster access to the 'healthcare-dataset' dataset.
Path to dataset files: /kaggle/input/healthcare-dataset
['healthcare_dataset.csv']


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


### Step 1b: Checking cardinality of every candidate column

In [8]:
candidate_cols = ['Gender', 'Blood Type', 'Medical Condition', 'Insurance Provider',
                   'Admission Type', 'Test Results', 'Doctor', 'Hospital', 'Name', 'Room Number']

for col in candidate_cols:
    print(f"{col}: {df[col].nunique()} distinct values")

Gender: 2 distinct values
Blood Type: 8 distinct values
Medical Condition: 6 distinct values
Insurance Provider: 5 distinct values
Admission Type: 3 distinct values
Test Results: 3 distinct values
Doctor: 40341 distinct values
Hospital: 39876 distinct values
Name: 49992 distinct values
Room Number: 400 distinct values


- The six columns from 2–8 distinct values are the clear case for lookup tables since they have low cardinality.

### Step 1c: Connect DuckDB in Colab

In [10]:
con = duckdb.connect('healthcare.duckdb')
print("Connected:", con)

Connected: <duckdb.duckdb.DuckDBPyConnection object at 0x7815e0b45c70>


### Step 1d: Creating Table Schemas

In [11]:
con.execute("CREATE OR REPLACE TABLE raw_admissions AS SELECT * FROM df")

# Gender
con.execute("""
CREATE OR REPLACE TABLE dim_gender (
    gender_id INTEGER PRIMARY KEY,
    gender VARCHAR NOT NULL UNIQUE
);
""")

# Blood Type
con.execute("""
CREATE OR REPLACE TABLE dim_blood_type (
    blood_type_id INTEGER PRIMARY KEY,
    blood_type VARCHAR NOT NULL UNIQUE
);
""")

# Medical Condition
con.execute("""
CREATE OR REPLACE TABLE dim_medical_condition (
    condition_id INTEGER PRIMARY KEY,
    condition_name VARCHAR NOT NULL UNIQUE
);
""")

# Insurance Provider
con.execute("""
CREATE OR REPLACE TABLE dim_insurance_provider (
    insurance_id INTEGER PRIMARY KEY,
    provider_name VARCHAR NOT NULL UNIQUE
);
""")

# Admission Type
con.execute("""
CREATE OR REPLACE TABLE dim_admission_type (
    admission_type_id INTEGER PRIMARY KEY,
    admission_type VARCHAR NOT NULL UNIQUE
);
""")

# Test Results
con.execute("""
CREATE OR REPLACE TABLE dim_test_results (
    test_result_id INTEGER PRIMARY KEY,
    test_result VARCHAR NOT NULL UNIQUE
);
""")

print("Empty table schemas created.")
con.execute("SHOW TABLES").fetchdf()

Empty table schemas created.


,name
0,dim_admission_type
1,dim_blood_type
2,dim_gender
3,dim_insurance_provider
4,dim_medical_condition
5,dim_test_results
6,raw_admissions


### Step 1e: Populate the lookup tables with distinct values

In [12]:
# Gender
con.execute("""
INSERT INTO dim_gender
SELECT ROW_NUMBER() OVER (ORDER BY Gender) AS gender_id, Gender
FROM (SELECT DISTINCT Gender FROM raw_admissions)
""")

# Blood Type
con.execute("""
INSERT INTO dim_blood_type
SELECT ROW_NUMBER() OVER (ORDER BY "Blood Type") AS blood_type_id, "Blood Type"
FROM (SELECT DISTINCT "Blood Type" FROM raw_admissions)
""")

# Medical Condition
con.execute("""
INSERT INTO dim_medical_condition
SELECT ROW_NUMBER() OVER (ORDER BY "Medical Condition") AS condition_id, "Medical Condition"
FROM (SELECT DISTINCT "Medical Condition" FROM raw_admissions)
""")

# Insurance Provider
con.execute("""
INSERT INTO dim_insurance_provider
SELECT ROW_NUMBER() OVER (ORDER BY "Insurance Provider") AS insurance_id, "Insurance Provider"
FROM (SELECT DISTINCT "Insurance Provider" FROM raw_admissions)
""")

# Admission Type
con.execute("""
INSERT INTO dim_admission_type
SELECT ROW_NUMBER() OVER (ORDER BY "Admission Type") AS admission_type_id, "Admission Type"
FROM (SELECT DISTINCT "Admission Type" FROM raw_admissions)
""")

# Test Results
con.execute("""
INSERT INTO dim_test_results
SELECT ROW_NUMBER() OVER (ORDER BY "Test Results") AS test_result_id, "Test Results"
FROM (SELECT DISTINCT "Test Results" FROM raw_admissions)
""")

print("Lookup tables populated.")

Lookup tables populated.


### Step 1f: Verify row counts and inspect contents

In [13]:
for tbl, expected in [('dim_gender', 2), ('dim_blood_type', 8), ('dim_medical_condition', 6),
                       ('dim_insurance_provider', 5), ('dim_admission_type', 3), ('dim_test_results', 3)]:
    count = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    status = "OK" if count == expected else "MISMATCH"
    print(f"{tbl}: {count} rows (expected {expected}) — {status}")

print()
for tbl in ['dim_gender', 'dim_blood_type', 'dim_medical_condition',
            'dim_insurance_provider', 'dim_admission_type', 'dim_test_results']:
    print(f"--- {tbl} ---")
    print(con.execute(f"SELECT * FROM {tbl}").fetchdf())
    print()

dim_gender: 2 rows (expected 2) — OK
dim_blood_type: 8 rows (expected 8) — OK
dim_medical_condition: 6 rows (expected 6) — OK
dim_insurance_provider: 5 rows (expected 5) — OK
dim_admission_type: 3 rows (expected 3) — OK
dim_test_results: 3 rows (expected 3) — OK

--- dim_gender ---
   gender_id  gender
0          1  Female
1          2    Male

--- dim_blood_type ---
   blood_type_id blood_type
0              1         A+
1              2         A-
2              3        AB+
3              4        AB-
4              5         B+
5              6         B-
6              7         O+
7              8         O-

--- dim_medical_condition ---
   condition_id condition_name
0             1      Arthritis
1             2         Asthma
2             3         Cancer
3             4       Diabetes
4             5   Hypertension
5             6        Obesity

--- dim_insurance_provider ---
   insurance_id     provider_name
0             1             Aetna
1             2        Blue Cr

### Step 2: Creating the fact_admissions Table Schema

In [14]:
con.execute("""
CREATE OR REPLACE TABLE fact_admissions (
    admission_id INTEGER PRIMARY KEY,
    patient_name VARCHAR,
    age INTEGER,
    gender_id INTEGER REFERENCES dim_gender(gender_id),
    blood_type_id INTEGER REFERENCES dim_blood_type(blood_type_id),
    condition_id INTEGER REFERENCES dim_medical_condition(condition_id),
    date_of_admission DATE,
    doctor VARCHAR,
    hospital VARCHAR,
    insurance_id INTEGER REFERENCES dim_insurance_provider(insurance_id),
    billing_amount DOUBLE,
    room_number INTEGER,
    admission_type_id INTEGER REFERENCES dim_admission_type(admission_type_id),
    discharge_date DATE,
    medication VARCHAR,
    test_result_id INTEGER REFERENCES dim_test_results(test_result_id)
);
""")

print("fact_admissions schema created.")
con.execute("SHOW TABLES").fetchdf()

fact_admissions schema created.


,name
0,dim_admission_type
1,dim_blood_type
2,dim_gender
3,dim_insurance_provider
4,dim_medical_condition
5,dim_test_results
6,fact_admissions
7,raw_admissions


### Step 2a: Populating fact_admissions with resolved foreign keys

In [16]:
con.execute("""
INSERT INTO fact_admissions
SELECT
    ROW_NUMBER() OVER () AS admission_id,
    r.Name AS patient_name,
    r.Age AS age,
    g.gender_id,
    bt.blood_type_id,
    mc.condition_id,
    r."Date of Admission" AS date_of_admission,
    r.Doctor AS doctor,
    r.Hospital AS hospital,
    ip.insurance_id,
    r."Billing Amount" AS billing_amount,
    r."Room Number" AS room_number,
    adm_t.admission_type_id,
    r."Discharge Date" AS discharge_date,
    r.Medication AS medication,
    tr.test_result_id
FROM raw_admissions r
JOIN dim_gender g ON r.Gender = g.gender
JOIN dim_blood_type bt ON r."Blood Type" = bt.blood_type
JOIN dim_medical_condition mc ON r."Medical Condition" = mc.condition_name
JOIN dim_insurance_provider ip ON r."Insurance Provider" = ip.provider_name
JOIN dim_admission_type adm_t ON r."Admission Type" = adm_t.admission_type
JOIN dim_test_results tr ON r."Test Results" = tr.test_result
""")

print("fact_admissions populated.")

raw_count = con.execute("SELECT COUNT(*) FROM raw_admissions").fetchone()[0]
fact_count = con.execute("SELECT COUNT(*) FROM fact_admissions").fetchone()[0]
print(f"raw_admissions: {raw_count} rows")
print(f"fact_admissions: {fact_count} rows")

fact_admissions populated.
raw_admissions: 55500 rows
fact_admissions: 55500 rows


### Step 3: A meaningful analytical query

In [17]:
query = """
SELECT
    mc.condition_name,
    ip.provider_name,
    COUNT(*) AS admission_count,
    ROUND(AVG(f.billing_amount), 2) AS avg_billing,
    ROUND(SUM(f.billing_amount), 2) AS total_billing
FROM fact_admissions f
JOIN dim_medical_condition mc ON f.condition_id = mc.condition_id
JOIN dim_insurance_provider ip ON f.insurance_id = ip.insurance_id
GROUP BY mc.condition_name, ip.provider_name
ORDER BY mc.condition_name, total_billing DESC
"""

result = con.execute(query).fetchdf()
print(result.to_string())

   condition_name     provider_name  admission_count  avg_billing  total_billing
0       Arthritis             Cigna             1900     25318.17    48104529.47
1       Arthritis  UnitedHealthcare             1873     25627.49    48000288.73
2       Arthritis        Blue Cross             1852     25792.79    47768244.54
3       Arthritis          Medicare             1851     25272.08    46778613.79
4       Arthritis             Aetna             1832     25478.95    46677443.70
5          Asthma             Cigna             1907     25610.47    48839168.72
6          Asthma  UnitedHealthcare             1870     25819.63    48282715.23
7          Asthma          Medicare             1833     25765.96    47229007.09
8          Asthma        Blue Cross             1835     25141.78    46135160.91
9          Asthma             Aetna             1740     25846.96    44973713.40
10         Cancer             Cigna             1864     25582.05    47684932.81
11         Cancer          M

### Step 4: Exporting the schema to a .sql file

In [20]:
schema_sql = """-- =====================================================================
-- Group 3 — Healthcare Dataset Schema (Lab 2)
-- DuckDB SQL schema: lookup/dimension tables + central fact table
--
-- Design decision: deliberate hybrid schema. Low-cardinality categorical
-- columns (2-8 distinct values, verified against the full 55,500-row
-- dataset) are normalized into lookup tables. High-cardinality columns
-- (Doctor, Hospital, Name — all 40,000+ distinct values) stay as plain
-- attributes in the fact table, since normalizing near-unique values
-- adds join overhead with no meaningful storage or consistency benefit.
-- See README.md for full reasoning.
-- =====================================================================

-- ---------------------------------------------------------------------
-- Raw staging table (loaded from source CSV via pandas -> DuckDB)
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE raw_admissions AS SELECT * FROM df;

-- ---------------------------------------------------------------------
-- Lookup / dimension tables — schema definitions
-- ---------------------------------------------------------------------

CREATE OR REPLACE TABLE dim_gender (
    gender_id INTEGER PRIMARY KEY,
    gender VARCHAR NOT NULL UNIQUE
);

CREATE OR REPLACE TABLE dim_blood_type (
    blood_type_id INTEGER PRIMARY KEY,
    blood_type VARCHAR NOT NULL UNIQUE
);

CREATE OR REPLACE TABLE dim_medical_condition (
    condition_id INTEGER PRIMARY KEY,
    condition_name VARCHAR NOT NULL UNIQUE
);

CREATE OR REPLACE TABLE dim_insurance_provider (
    insurance_id INTEGER PRIMARY KEY,
    provider_name VARCHAR NOT NULL UNIQUE
);

CREATE OR REPLACE TABLE dim_admission_type (
    admission_type_id INTEGER PRIMARY KEY,
    admission_type VARCHAR NOT NULL UNIQUE
);

CREATE OR REPLACE TABLE dim_test_results (
    test_result_id INTEGER PRIMARY KEY,
    test_result VARCHAR NOT NULL UNIQUE
);

-- ---------------------------------------------------------------------
-- Lookup / dimension tables — populate with distinct values
-- ---------------------------------------------------------------------

INSERT INTO dim_gender
SELECT ROW_NUMBER() OVER (ORDER BY Gender) AS gender_id, Gender
FROM (SELECT DISTINCT Gender FROM raw_admissions);

INSERT INTO dim_blood_type
SELECT ROW_NUMBER() OVER (ORDER BY "Blood Type") AS blood_type_id, "Blood Type"
FROM (SELECT DISTINCT "Blood Type" FROM raw_admissions);

INSERT INTO dim_medical_condition
SELECT ROW_NUMBER() OVER (ORDER BY "Medical Condition") AS condition_id, "Medical Condition"
FROM (SELECT DISTINCT "Medical Condition" FROM raw_admissions);

INSERT INTO dim_insurance_provider
SELECT ROW_NUMBER() OVER (ORDER BY "Insurance Provider") AS insurance_id, "Insurance Provider"
FROM (SELECT DISTINCT "Insurance Provider" FROM raw_admissions);

INSERT INTO dim_admission_type
SELECT ROW_NUMBER() OVER (ORDER BY "Admission Type") AS admission_type_id, "Admission Type"
FROM (SELECT DISTINCT "Admission Type" FROM raw_admissions);

INSERT INTO dim_test_results
SELECT ROW_NUMBER() OVER (ORDER BY "Test Results") AS test_result_id, "Test Results"
FROM (SELECT DISTINCT "Test Results" FROM raw_admissions);

-- ---------------------------------------------------------------------
-- Central fact table — schema definition
-- Grain: one row per hospital admission
-- ---------------------------------------------------------------------

CREATE OR REPLACE TABLE fact_admissions (
    admission_id INTEGER PRIMARY KEY,
    patient_name VARCHAR,
    age INTEGER,
    gender_id INTEGER REFERENCES dim_gender(gender_id),
    blood_type_id INTEGER REFERENCES dim_blood_type(blood_type_id),
    condition_id INTEGER REFERENCES dim_medical_condition(condition_id),
    date_of_admission DATE,
    doctor VARCHAR,
    hospital VARCHAR,
    insurance_id INTEGER REFERENCES dim_insurance_provider(insurance_id),
    billing_amount DOUBLE,
    room_number INTEGER,
    admission_type_id INTEGER REFERENCES dim_admission_type(admission_type_id),
    discharge_date DATE,
    medication VARCHAR,
    test_result_id INTEGER REFERENCES dim_test_results(test_result_id)
);

-- ---------------------------------------------------------------------
-- Central fact table — populate, resolving text values to foreign keys
-- via joins against each lookup table
-- ---------------------------------------------------------------------

INSERT INTO fact_admissions
SELECT
    ROW_NUMBER() OVER () AS admission_id,
    r.Name AS patient_name,
    r.Age AS age,
    g.gender_id,
    bt.blood_type_id,
    mc.condition_id,
    r."Date of Admission" AS date_of_admission,
    r.Doctor AS doctor,
    r.Hospital AS hospital,
    ip.insurance_id,
    r."Billing Amount" AS billing_amount,
    r."Room Number" AS room_number,
    adm_t.admission_type_id,
    r."Discharge Date" AS discharge_date,
    r.Medication AS medication,
    tr.test_result_id
FROM raw_admissions r
JOIN dim_gender g ON r.Gender = g.gender
JOIN dim_blood_type bt ON r."Blood Type" = bt.blood_type
JOIN dim_medical_condition mc ON r."Medical Condition" = mc.condition_name
JOIN dim_insurance_provider ip ON r."Insurance Provider" = ip.provider_name
JOIN dim_admission_type adm_t ON r."Admission Type" = adm_t.admission_type
JOIN dim_test_results tr ON r."Test Results" = tr.test_result;

-- ---------------------------------------------------------------------
-- Sample analytical query: admission count and billing totals by
-- medical condition and insurance provider (supports the cost-planning
-- and billing-accuracy decisions identified in the Data Problem Statement)
-- ---------------------------------------------------------------------

SELECT
    mc.condition_name,
    ip.provider_name,
    COUNT(*) AS admission_count,
    ROUND(AVG(f.billing_amount), 2) AS avg_billing,
    ROUND(SUM(f.billing_amount), 2) AS total_billing
FROM fact_admissions f
JOIN dim_medical_condition mc ON f.condition_id = mc.condition_id
JOIN dim_insurance_provider ip ON f.insurance_id = ip.insurance_id
GROUP BY mc.condition_name, ip.provider_name
ORDER BY mc.condition_name, total_billing DESC;
"""

with open("patients_schema.sql", "w") as f:
    f.write(schema_sql)

print("patients_schema.sql written.")
print("Location:", os.path.abspath("patients_schema.sql"))

patients_schema.sql written.
Location: /content/patients_schema.sql


### Step 5: Downloading patients_schema.sql

In [21]:
files.download("patients_schema.sql")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>